# Check GCS TIF Files - COG Status

Simple check to see if GCS TIF files are Cloud-Optimized GeoTIFFs (COGs)

In [ ]:
# Cell 1: Check GCP credentials
from google.cloud import storage

try:
    client = storage.Client()
    print(f"✅ GCP credentials OK")
    print(f"   Project: {client.project}")
except Exception as e:
    print(f"❌ GCP credentials failed: {e}")
    print("   Run: gcloud auth application-default login")

In [ ]:
# Cell 2: Load global_inputs.yml and extract TIF file paths
from google.cloud import storage
import yaml

# Download global_inputs.yml from GCS (this is what main.py does)
INPUT_BUCKET = 'crp-city-scan'
INPUT_DIR = '01-user-input'
DATA_BUCKET = 'city-scan-global-data'

client = storage.Client()
bucket = client.bucket(INPUT_BUCKET)
blob = bucket.blob(f'{INPUT_DIR}/global_inputs.yml')

# Download to temp location
import tempfile
with tempfile.NamedTemporaryFile(mode='w+', suffix='.yml', delete=False) as f:
    blob.download_to_filename(f.name)
    temp_path = f.name

# Load global inputs
with open(temp_path, 'r') as f:
    global_inputs = yaml.safe_load(f)

# Extract all TIF-related entries
gcs_tifs = {}
for key, value in global_inputs.items():
    if isinstance(value, str) and '.tif' in value.lower():
        # Construct full GCS path
        if value.startswith('gs://'):
            gcs_path = value
        else:
            gcs_path = f'gs://{DATA_BUCKET}/{value}'
        
        gcs_tifs[key] = gcs_path

print(f"Found {len(gcs_tifs)} TIF references in global_inputs.yml:\n")
for key, path in sorted(gcs_tifs.items()):
    print(f"  {key}: {path}")

In [ ]:
# Cell 3: Check each TIF if it's a COG
import rasterio

results = []

for name, gcs_path in gcs_tifs.items():
    vsi_path = gcs_path.replace('gs://', '/vsigs/')
    
    print(f"Checking {name}... ", end='')
    
    try:
        with rasterio.open(vsi_path) as src:
            is_tiled = src.profile.get('tiled', False)
            has_overviews = len(src.overviews(1)) > 0 if src.count > 0 else False
            blockx = src.profile.get('blockxsize', 0)
            blocky = src.profile.get('blockysize', 0)
            
            is_cog = is_tiled and has_overviews and blockx >= 256 and blocky >= 256
            
            results.append({
                'name': name,
                'path': gcs_path,
                'is_cog': is_cog,
                'tiled': is_tiled,
                'overviews': has_overviews,
                'block_size': f"{blockx}x{blocky}",
                'error': None
            })
            
            status = "✅ COG" if is_cog else "❌ NOT COG"
            print(f"{status} (tiled={is_tiled}, overviews={has_overviews}, blocks={blockx}x{blocky})")
            
    except Exception as e:
        results.append({
            'name': name,
            'path': gcs_path,
            'is_cog': False,
            'tiled': None,
            'overviews': None,
            'block_size': None,
            'error': str(e)
        })
        print(f"❌ ERROR: {str(e)[:80]}")

# Summary
print("\n" + "="*80)
cogs = sum(1 for r in results if r['is_cog'])
errors = sum(1 for r in results if r['error'])
print(f"✅ COGs: {cogs}/{len(results)}")
print(f"❌ Not COGs: {len(results) - cogs - errors}/{len(results)}")
print(f"⚠️  Errors: {errors}/{len(results)}")
print("="*80)